In [1]:
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from collections import Counter

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" # to show output from all the lines in a cells
pd.set_option('display.max_column',None) # display all the columns in pandas
pd.options.display.max_rows = 100

from datetime import date
today = str(date.today())

In [2]:
import scanpy as sc

In [3]:
import dandelion as ddl

/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/lib/python3.14/site-packages/nxviz/__init__.py:33: UserWarning: 
nxviz has a new API! Version 0.7.4 onwards, the old class-based API is being
deprecated in favour of a new API focused on advancing a grammar of network
graphics. If your plotting code depends on the old API, please consider
pinning nxviz at version 0.7.4, as the new API will break your old code.

To check out the new API, please head over to the docs at
https://ericmjl.github.io/nxviz/ to learn more. We hope you enjoy using it!

(This deprecation message will go away in version 1.0.)



In [4]:
print(ddl.__version__)

0.5.7


In [5]:
import scirpy as ir

/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [6]:
import os, shutil, sys

print(sys.executable)
print(os.environ.get("R_HOME"))
print(shutil.which("R"))
print(os.environ["PATH"])

/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/bin/python
/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/lib/R
/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/bin/R
/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin


In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
import rpy2.robjects as ro

print(ro.r("R.version.string")[0])
print(ro.r("R.home()")[0])
print(ro.r(".libPaths()"))

R version 4.5.3 (2026-03-11)
/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/lib/R
[1] "/mnt/nfs/home/anamaceiras/miniforge3/envs/ddl/lib/R/library"



In [9]:
import rpy2.rinterface_lib.callbacks
%load_ext rpy2.ipython

## TCR Reannotation

In [ ]:
personal_folder = 'anamaceiras' # <- add you folder name

In [10]:
# change directory to folder with metadata
os.chdir(os.path.expanduser('/mnt/nfs/home/'+personal_folder+'/BCR_TCR/TCR_BCR_data'))
# print current working directory
os.getcwd()

'/mnt/nfs/home/anamaceiras/scCourse_files'

### Samples list

In [11]:
metadata = pd.read_csv('samples_list.csv', index_col=0)

In [12]:
metadata = metadata[metadata.donor_list=='694B'] #either 694B, 759B, or 778C

In [13]:
metadata.head(2)

,sample_library_id,donor_list
0,694B_001_BCR_CZI-IA11485873,694B
2,694B_001_BCR_CZI-IA11512685,694B


In [14]:
samples = metadata['sample_library_id']
len(samples)

14

In [15]:
donor_list = list(set(metadata.donor_list))
donor_list.sort()
donor_list

['694B']

In [16]:
TCR_samples=[]
for s in samples:
    if 'TCR' in s:
        TCR_samples.append(s)
#print(TCR_samples)
#print(BCR_samples)

In [17]:
len(TCR_samples)

7

In [18]:
TCR_samples.sort()
TCR_samples[:5]

['694B_001_TCR_CZI-IA11485873',
 '694B_001_TCR_CZI-IA11512685',
 '694B_001_TCR_CZI-IA11512686',
 '694B_001_TCR_CZI-IA11512687',
 '694B_001_TCR_CZI-IA11512688']

In [19]:
TCRsamples_prefix=[x.split('_')[3] for x in TCR_samples]
TCRsamples_prefix[:5]

['CZI-IA11485873',
 'CZI-IA11512685',
 'CZI-IA11512686',
 'CZI-IA11512687',
 'CZI-IA11512688']

### Change working directory

### Step 1: Formatting the headers of the Cell Ranger fasta file

In [22]:
# the first option is a list of fasta files to format and the second option is the list of prefix to add to each file.
ddl.pp.format_fastas(TCR_samples, filename_prefix = [x+'.cellranger.all' for x in TCR_samples], prefix=TCRsamples_prefix)

Formatting fasta(s) : 100%|██████████| 7/7 [00:01<00:00,  3.87it/s]


### Step 2: Reannotate the DJ genes with igblastn

In [24]:
ddl.pp.reannotate_genes(TCR_samples, loci = 'tr', reassign_dj = True, 
                        filename_prefix = [x+'.cellranger.all' for x in TCR_samples])

Assigning genes :   0%|          | 0/7 [00:00<?, ?it/s]

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11485873.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11485873.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:29:15 |Done                | 0.0 min

PROGRESS> 14:29:19 |####################| 100% (10,843) 0.1 min

OUTPUT> 694B_001_TCR_CZI-IA11485873.cellranger.all_contig_igblast_db-pass.tsv
  PASS> 8342
  FAIL> 2501
   END> MakeDb

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11485873.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11485873.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:29:19 |Done                | 0.0 min

PROGRESS> 14:29:23 |####################| 100% (10,843) 0

Assigning genes :  14%|█▍        | 1/7 [01:42<10:14, 102.44s/it]

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512685.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512685.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:31:59 |Done                | 0.0 min

PROGRESS> 14:32:05 |####################| 100% (17,302) 0.1 min

OUTPUT> 694B_001_TCR_CZI-IA11512685.cellranger.all_contig_igblast_db-pass.tsv
  PASS> 15499
  FAIL> 1803
   END> MakeDb

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512685.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512685.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:32:06 |Done                | 0.0 min

PROGRESS> 14:32:12 |####################| 100% (17,302) 

Assigning genes :  29%|██▊       | 2/7 [04:41<12:18, 147.77s/it]

...

Assigning genes :  71%|███████▏  | 5/7 [11:12<04:11, 125.81s/it]

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512689.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512689.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:39:39 |Done                | 0.0 min

PROGRESS> 14:39:40 |####################| 100% (3,453) 0.0 min

OUTPUT> 694B_001_TCR_CZI-IA11512689.cellranger.all_contig_igblast_db-pass.tsv
  PASS> 3101
  FAIL> 352
   END> MakeDb

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512689.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512689.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:39:41 |Done                | 0.0 min

PROGRESS> 14:39:42 |####################| 100% (3,453) 0.0 

Assigning genes :  86%|████████▌ | 6/7 [11:50<01:36, 96.03s/it] 

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512690.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512690.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:40:40 |Done                | 0.0 min

PROGRESS> 14:40:43 |####################| 100% (6,240) 0.0 min

OUTPUT> 694B_001_TCR_CZI-IA11512690.cellranger.all_contig_igblast_db-pass.tsv
  PASS> 5601
  FAIL> 639
   END> MakeDb

         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> 694B_001_TCR_CZI-IA11512690.cellranger.all_contig_igblast.fmt7
      SEQ_FILE> 694B_001_TCR_CZI-IA11512690.cellranger.all_contig.fasta
         NPROC> 64
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> True
INFER_JUNCTION> False

PROGRESS> 14:40:43 |Done                | 0.0 min

PROGRESS> 14:40:46 |####################| 100% (6,240) 0.0 

Assigning genes : 100%|██████████| 7/7 [12:58<00:00, 111.20s/it]


### Concatenate TCR reannotated data

In [25]:
tcr_files = []
c=0
for sample in TCR_samples:
    c+=1
    #print(sample)
    file_location = sample +'/dandelion/' + sample + '.cellranger.all_contig_dandelion.tsv'
    tcr_files.append(pd.read_csv(file_location, sep = '\t'))

In [27]:
tcr = pd.concat(tcr_files)
tcr.reset_index(inplace = True, drop = True)
tcr

,sequence_id,sequence,rev_comp,productive,v_call,d_call,j_call,sequence_alignment,germline_alignment,junction,junction_aa,v_cigar,d_cigar,j_cigar,stop_codon,vj_in_frame,locus,c_call,junction_length,np1_length,np2_length,v_sequence_start,v_sequence_end,v_germline_start,v_germline_end,d_sequence_start,d_sequence_end,d_germline_start,d_germline_end,j_sequence_start,j_sequence_end,j_germline_start,j_germline_end,v_score,v_identity,v_support,d_score,d_identity,d_support,j_score,j_identity,j_support,fwr1,fwr2,fwr3,fwr4,cdr1,cdr2,cdr3,cell_id,consensus_count,umi_count,v_call_10x,d_call_10x,j_call_10x,junction_10x,junction_10x_aa,j_support_igblastn,j_score_igblastn,j_call_igblastn,j_call_blastn,j_identity_blastn,j_alignment_length_blastn,j_number_of_mismatches_blastn,j_number_of_gap_openings_blastn,j_sequence_start_blastn,j_sequence_end_blastn,j_germline_start_blastn,j_germline_end_blastn,j_support_blastn,j_score_blastn,j_sequence_alignment_blastn,j_germline_alignment_blastn,j_source,d_support_igblastn,d_score_igblastn,d_call_igblastn,d_call_blastn,d_identity_blastn,d_alignment_length_blastn,d_number_of_mismatches_blastn,d_number_of_gap_openings_blastn,d_sequence_start_blastn,d_sequence_end_blastn,d_germline_start_blastn,d_germline_end_blastn,d_support_blastn,d_score_blastn,d_sequence_alignment_blastn,d_germline_alignment_blastn,d_source,junction_aa_length,fwr1_aa,fwr2_aa,fwr3_aa,fwr4_aa,cdr1_aa,cdr2_aa,cdr3_aa,sequence_alignment_aa,v_sequence_alignment_aa,d_sequence_alignment_aa,j_sequence_alignment_aa,j_call_multimappers,j_call_multiplicity,j_call_sequence_start_multimappers,j_call_sequence_end_multimappers,j_call_support_multimappers
0,CZI-IA11485873_AAACCTGAGAAGGTTT_contig_1,AGACCAGAATCCTGCCCTGGGCCTTGCCTGGTCTGCCTCACTCTGC...,F,T,TRBV3-1*01,TRBD1*01,TRBJ1-2*01,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,TGTGCCAGCAGCCTGACAGGGGGCCTGCAGTATGGCTACACCTTC,CASSLTGGLQYGYTF,104S283=,388S2N10=,404S5N43=,F,T,TRB,TRBC1,45,1,6.0,105,387,1,322,389.0,398.0,3.0,12.0,405,447,6,48,442.0,1.0,1.740000e-126,20.4,1.0,0.014000,83.4,1.0,1.770000e-19,GACACAGCTGTTTCCCAGACTCCAAAATACCTGGTCACACAGATGG...,ATGTATTGGTATAAACAGGACTCTAAGAAATTTCTGAAGATAATGT...,ATTATAAATGAAACAGTTCCA...AATCGCTTCTCACCTAAATCTC...,TTCGGTTCGGGGACCAGGTTAACCGTTGTAG,CTGGGCCAT.....................GATACT,TACAATAAT............AAGGAGCTC,GCCAGCAGCCTGACAGGGGGCCTGCAGTATGGCTACACC,CZI-IA11485873_AAACCTGAGAAGGTTT,46,1,TRBV3-1,TRBD1,TRBJ1-2,TGTGCCAGCAGCCTGACAGGGGGCCTGCAGTATGGCTACACCTTC,CASSLTGGLQYGYTF,1.770000e-19,83.4,TRBJ1-2*01,TRBJ1-2*01,100.0,43.0,0.0,0.0,405.0,447.0,6.0,48.0,1.200000e-18,80.5,TATGGCTACACCTTCGGTTCGGGGACCAGGTTAACCGTTGTAG,TATGGCTACACCTTCGGTTCGGGGACCAGGTTAACCGTTGTAG,NaN,0.014000,20.4,TRBD1*01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,DTAVSQTPKYLVTQMGNDKSIKCEQN,MYWYKQDSKKFLKIMFS,IINETVPNRFSPKSPDKAHLNLHINSLELGDSAVYFC,FGSGTRLTVV,LGHDT,YNNKEL,ASSLTGGLQYGYT,DTAVSQTPKYLVTQMGNDKSIKCEQNLGHDTMYWYKQDSKKFLKIM...,DTAVSQTPKYLVTQMGNDKSIKCEQNLGHDTMYWYKQDSKKFLKIM...,TGG,YGYTFGSGTRLTVV,TRBJ1-2*01,1.0,405,447,1.2e-18
1,CZI-IA11485873_AAACCTGAGGGATCTG_contig_1,ATTCTTTCTTCAAAGCAGCCATGGGAATCAGGCTCCTCTGTCGTGT...,F,T,TRBV28*01,NaN,TRBJ1-4*01,GATGTGAAAGTAACCCAGAGCTCGAGATATCTAGTCAAAAGGACGG...,GATGTGAAAGTAACCCAGAGCTCGAGATATCTAGTCAAAAGGACGG...,TGTGCCAGCAGTTTCCTCCGGGGGGAAAAACTGTTTTTT,CASSFLRGEKLFF,77S284=,NaN,371S8N43=,F,T,TRB,TRBC1,39,10,NaN,78,361,1,323,NaN,NaN,NaN,NaN,372,414,9,51,444.0,1.0,5.510000e-127,NaN,NaN,NaN,83.4,1.0,1.650000e-19,GATGTGAAAGTAACCCAGAGCTCGAGATATCTAGTCAAAAGGACGG...,ATGTTCTGGTATCGACAAGACCCAGGTCTGGGGCTACGGCTGATCT...,AAAGAAAAAGGAGATATTCCT...GAGGGGTACAGTGTCTCTAGAG...,TTTGGCAGTGGAACCCAGCTCTCTGTCTTGG,ATGGACCAT.....................GAAAAT,TCATATGAT............GTTAAAATG,GCCAGCAGTTTCCTCCGGGGGGAAAAACTGTTT,CZI-IA11485873_AAACCTGAGGGATCTG,4219,5,TRBV28,NaN,TRBJ1-4,TGTGCCAGCAGTTTCCTCCGGGGGGAAAAACTGTTTTTT,CASSFLRGEKLFF,1.650000e-19,83.4,TRBJ1-4*01,TRBJ1-4*01,100.0,43.0,0.0,0.0,372.0,414.0,9.0,51.0,1.120000e-18,80.5,GAAAAACTGTTTTTTGGCA

In [28]:
tcr.to_csv('/mnt/nfs/home/'+personal_folder+'/BCR_TCR/TCR_BCR_data/TCR_concatenated_ddl.csv')